In [2]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path.cwd().parent
sys.path.append(str(project_root))

print(project_root)

c:\Users\Satvika Putcha\OneDrive\Dokumen\equity-risk-assessment-framework


In [3]:
daily_returns = pd.read_csv(
    project_root / "data" / "processed" / "daily_returns.csv",
    index_col=0,
    parse_dates=True
)

daily_returns.head()

,JPM,SPY,TLT,GLD,JNJ,AAPL,XOM,KO
Date,,,,,,,,
2016-01-05,0.001729,0.001691,-0.004034,0.002819,0.004180,-0.025059,0.008520,0.003538
2016-01-06,-0.014436,-0.012614,0.013475,0.014441,-0.005055,-0.019570,-0.008320,-0.005406
2016-01-07,-0.040439,-0.023991,0.001795,0.014140,-0.011654,-0.042205,-0.016006,-0.016540
2016-01-08,-0.022399,-0.010977,0.004479,-0.004428,-0.010684,0.005288,-0.020202,-0.002643
2016-01-11,-0.001528,0.000990,-0.010944,-0.008895,-0.006010,0.016192,-0.013389,0.001686


Annual Returns

In [4]:
annual_returns = (1+daily_returns).prod() ** (252 / len(daily_returns)) -1

annual_returns

JPM     0.206839
SPY     0.151494
TLT    -0.007847
GLD     0.131987
JNJ     0.123151
AAPL    0.282896
XOM     0.114620
KO      0.098636
dtype: float64

Annual Volatility

In [5]:
annual_volatility = daily_returns.std() * np.sqrt(252)

annual_volatility

JPM     0.274040
SPY     0.178086
TLT     0.147469
GLD     0.162449
JNJ     0.184832
AAPL    0.289163
XOM     0.279524
KO      0.181695
dtype: float64

Sharpe Ratio

In [6]:
sharpe_ratio = annual_returns / annual_volatility

sharpe_ratio

JPM     0.754775
SPY     0.850677
TLT    -0.053212
GLD     0.812481
JNJ     0.666285
AAPL    0.978327
XOM     0.410056
KO      0.542866
dtype: float64

Correlation Matrix

In [7]:
correlation_matrix = daily_returns.corr()
correlation_matrix

,JPM,SPY,TLT,GLD,JNJ,AAPL,XOM,KO
JPM,1.000000,0.701697,-0.307057,-0.063800,0.329744,0.407726,0.503279,0.385279
SPY,0.701697,1.000000,-0.155990,0.076342,0.428062,0.741368,0.486361,0.491976
TLT,-0.307057,-0.155990,1.000000,0.259906,-0.086755,-0.092116,-0.237250,-0.071798
GLD,-0.063800,0.076342,0.259906,1.000000,0.043699,0.035133,0.036421,0.044607
JNJ,0.329744,0.428062,-0.086755,0.043699,1.000000,0.289434,0.283459,0.494959
AAPL,0.407726,0.741368,-0.092116,0.035133,0.289434,1.000000,0.277226,0.323400
XOM,0.503279,0.486361,-0.237250,0.036421,0.283459,0.277226,1.000000,0.355110
KO,0.385279,0.491976,-0.071798,0.044607,0.494959,0.323400,0.355110,1.000000


Portfolio Returns

In [8]:
weights = np.array([1/8] * 8)

portfolio_returns = daily_returns.dot(weights)

portfolio_returns.head()

Date
2016-01-05   -0.000827
2016-01-06   -0.004686
2016-01-07   -0.016863
2016-01-08   -0.007696
2016-01-11   -0.002737
dtype: float64

Portfolio Annual Return

In [9]:
portfolio_annual_return = (1 + portfolio_returns).prod() ** (252 / len(portfolio_returns)) -1

print(f"Portfolio Annual Returns: {portfolio_annual_return:.2%}")


Portfolio Annual Returns: 15.34%


Portfolio Annual Volatility

In [10]:
portfolio_annual_volatility = portfolio_returns.std() * np.sqrt(252)

print(f"Porfolio Annual Volatility: {portfolio_annual_volatility:.2%}")

Porfolio Annual Volatility: 12.58%


Portfolio Sharpe Ratio

In [11]:
portfolio_sharpe = portfolio_annual_return / portfolio_annual_volatility

print(f"Portfolio Sharpe Ratio: {portfolio_sharpe:.3f}")

Portfolio Sharpe Ratio: 1.219


Portfolio Value

In [12]:
portfolio_value = (1 + portfolio_returns).cumprod()

portfolio_value.head()

Date
2016-01-05    0.999173
2016-01-06    0.994491
2016-01-07    0.977721
2016-01-08    0.970197
2016-01-11    0.967542
dtype: float64

Running Maximum

In [13]:
running_max = portfolio_value.cummax()

running_max.head()

Date
2016-01-05    0.999173
2016-01-06    0.999173
2016-01-07    0.999173
2016-01-08    0.999173
2016-01-11    0.999173
dtype: float64

Drawdown

In [14]:
drawdown = (portfolio_value - running_max) / running_max

drawdown.head()

Date
2016-01-05    0.000000
2016-01-06   -0.004686
2016-01-07   -0.021469
2016-01-08   -0.029000
2016-01-11   -0.031657
dtype: float64

Maximum Drawdown

In [15]:
max_drawdown = drawdown.min()

print(f"Maximum Drawdown: {max_drawdown:.2%}")

Maximum Drawdown: -26.93%


Value at Risk

In [16]:
var_95 = portfolio_returns.quantile(0.05)

print(f"95% Daily VaR: {var_95:.2%}")

95% Daily VaR: -1.09%


Load Market Returns(SPY)

In [17]:
market_returns = daily_returns["SPY"]

market_returns.head()

Date
2016-01-05    0.001691
2016-01-06   -0.012614
2016-01-07   -0.023991
2016-01-08   -0.010977
2016-01-11    0.000990
Name: SPY, dtype: float64

In [18]:
type(market_returns)

pandas.core.series.Series

Market Variance

In [19]:
market_variance = market_returns.var()

print("Market Variance: ",market_variance)

Market Variance:  0.00012585169877138506


Beta for each Asset

In [20]:
beta = {}

for asset in daily_returns.columns:
    covariance = daily_returns[asset].cov(market_returns)
    beta[asset] = covariance / market_variance

beta = pd.Series(beta)

beta

JPM     1.079778
SPY     1.000000
TLT    -0.129172
GLD     0.069639
JNJ     0.444277
AAPL    1.203777
XOM     0.763391
KO      0.501946
dtype: float64

Asset Summary Table

In [21]:
asset_summary = pd.DataFrame({
    "Annual Return": annual_returns,
    "Annual Volatility": annual_volatility,
    "Sharpe Ratio": sharpe_ratio,
    "Beta": beta
})

asset_summary

,Annual Return,Annual Volatility,Sharpe Ratio,Beta
JPM,0.206839,0.274040,0.754775,1.079778
SPY,0.151494,0.178086,0.850677,1.000000
TLT,-0.007847,0.147469,-0.053212,-0.129172
GLD,0.131987,0.162449,0.812481,0.069639
JNJ,0.123151,0.184832,0.666285,0.444277
AAPL,0.282896,0.289163,0.978327,1.203777
XOM,0.114620,0.279524,0.410056,0.763391
KO,0.098636,0.181695,0.542866,0.501946


Save Asset Summary

In [22]:
asset_summary.index.name = "Asset"

asset_summary.to_csv(
    project_root / "data" / "processed" / "asset_summary.csv"
)

print("asset_summary.csv saved!")

asset_summary.csv saved!


Export Correlation Matrix

In [23]:
correlation_matrix.to_csv(
    project_root / "data" / "processed" / "correlation_matrix.csv"
)

print("correlation_matrix.csv saved!")

correlation_matrix.csv saved!


Create Portfolio History

In [24]:
portfolio_history = pd.DataFrame({
    "Portfolio Value": portfolio_value,
    "Running Max": running_max,
    "Drawdown": drawdown
})

portfolio_history.head()

,Portfolio Value,Running Max,Drawdown
Date,,,
2016-01-05,0.999173,0.999173,0.000000
2016-01-06,0.994491,0.999173,-0.004686
2016-01-07,0.977721,0.999173,-0.021469
2016-01-08,0.970197,0.999173,-0.029000
2016-01-11,0.967542,0.999173,-0.031657


Export Portfolio History

In [25]:
portfolio_history.to_csv(
    project_root / "data" / "processed" / "portfolio_history.csv"
)

print("portfolio_history.csv saved!")

portfolio_history.csv saved!


Create Portfolio SUmmary

In [26]:
portfolio_summary = pd.DataFrame({
    "Annual Return": [portfolio_annual_return],
    "Annual Volatility": [portfolio_annual_volatility],
    "Sharpe Ratio": [portfolio_sharpe],
    "Maximum Drawdown": [max_drawdown],
    "95% VaR": [var_95]
})

portfolio_summary

,Annual Return,Annual Volatility,Sharpe Ratio,Maximum Drawdown,95% VaR
0,0.15338,0.125813,1.219116,-0.26927,-0.010934


Export Portfolio Summary

In [27]:
portfolio_summary.to_csv(
    project_root / "data" / "processed" / "portfolio_summary.csv",
    index=False
)

print("portfolio_summary.csv saved!")

portfolio_summary.csv saved!


Power bi correlation heatmap

In [28]:
correlation_long = (
    correlation_matrix
    .reset_index()
    .rename(columns={"index": "Asset1"})
    .melt(
        id_vars = "Asset1",
        var_name = "Asset2",
        value_name = "Correlation"
    )
)

correlation_long.head()

,Asset1,Asset2,Correlation
0,JPM,JPM,1.000000
1,SPY,JPM,0.701697
2,TLT,JPM,-0.307057
3,GLD,JPM,-0.063800
4,JNJ,JPM,0.329744


In [29]:
correlation_long.to_csv(
    project_root / "data" / "processed" / "correlation_long.csv",
    index=False
)

print("saved!")


saved!
